In [ ]:
# Ruta a la carpeta del proyecto (una carpeta "arriba" de notebooks/)
RUTA = ".."


In [ ]:
import pandas as pd
import requests

url = "https://apidatos.ree.es/es/datos/demanda/demanda-tiempo-real"

# Primero definimos los parámetros...
params = {
    "start_date": "2026-10-01T00:00",
    "end_date": "2026-10-07T23:59",
    "time_trunc": "hour"
}

# ...y después hacemos la petición
r = requests.get(url, params=params)
print(r.status_code)
datos = r.json()
datos.keys()

# ESQUEMA QUE VAMOS A SEGUIR
1. CONSEGUIR DATOS
   Demanda eléctrica real de España (REE) + temperatura (Open-Meteo)
        
2. LIMPIAR
   Pasar de 5 minutos a horas, arreglar el cambio de hora, huecos…
        
3. EXPLORAR
   ¿Se consume más a las 20:00? ¿En agosto? ¿Los domingos? ¿Con calor?
        
4. CREAR VARIABLES
   hora, día de la semana, festivo, temperatura, consumo de ayer…
        
5. ENTRENAR EL MODELO
   El modelo aprende: "lunes, 20:00, 35 °C → unos 38.000 MW"
        
6. EVALUAR
   ¿Predigo mejor o peor que Red Eléctrica?


# CONSEGUIR DATOS
Demanda eléctrica real de España (REE) + temperatura (Open-Meteo)




In [ ]:
type(datos["included"][0]["attributes"])# NIVEL 1: datos es un dict, miramos sus claves
print(type(datos))
print(datos.keys())

# NIVEL 2: "included" es una lista, miramos cuántos elementos tiene
print(type(datos["included"]))
print(len(datos["included"]))   # 4

# NIVEL 3: el primer elemento es un dict
print(type(datos["included"][0]))

# NIVEL 4: dentro tiene "attributes", otro dict, miramos sus claves
print(type(datos["included"][0]["attributes"]))
print(datos["included"][0]["attributes"].keys())



In [ ]:
#Comprobamos que tenemos razon
print(datos["included"][0]["attributes"]["title"])        # el nombre
print(len(datos["included"][0]["attributes"]["values"]))   # cuántos valores hay
print(datos["included"][0]["attributes"]["values"][0])     # el primer valor


In [ ]:
# Hacemos bucle para evr cual de las 4 series es la real
for serie in datos["included"]:
    if serie["attributes"]["title"] == "Real":
        valores_real = serie["attributes"]["values"]

print(len(valores_real))



In [ ]:
df= pd.DataFrame(valores_real)
df.head()
df = df[["datetime", "value"]]
# Renombramos "value" a algo más claro {"nombre_viejo": "nombre_nuevo"}
df = df.rename(columns={"value": "demanda_mw"})

print(df.head())


In [ ]:
# Cuantas filas tiene df
df.info()
# Demanda máxima y mínima (podriamos utilizar df.describe() sin embargo como solo nos ineteresa el max y min utilizamos  uno en concreto)
print(df["demanda_mw"].max())
print(df["demanda_mw"].min())


In [ ]:
# A) Convertir el texto en fecha  y luego a hora española
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df["datetime"] = df["datetime"].dt.tz_convert("Europe/Madrid")

print(df.dtypes)   # datetime64[ns, Europe/Madrid]
# B) Ahora sí: extraer hora y día de la semana
df["hora"] = df["datetime"].dt.hour
df["dia_semana"] = df["datetime"].dt.dayofweek   # 0 = lunes, 6 = domingo (es decir: 1 = martes, 2 = miercoles, 3 = jueves, 4 = viernes, 5 = sábado)
df["mes"] = df["datetime"].dt.month
df.head()


In [ ]:
# Defino función
def descargar_semana(inicio, fin):
    # 1. Preparar los parámetros (inicio y fin son variables: no comillas)
    params = {
        "start_date": inicio,
        "end_date": fin,
        "time_trunc": "hour"
    }

    # 2. Hacer la petición y convertir la respuesta a diccionario
    r = requests.get(url, params=params)
    datos = r.json()

    # 3. Buscar la serie "Real"
    for serie in datos["included"]:
        if serie["attributes"]["title"] == "Real":
            valores_real = serie["attributes"]["values"]

    # 4. Crear la tabla y quedarnos con lo que necesitamos
    df = pd.DataFrame(valores_real)
    df = df[["datetime", "value"]]
    df = df.rename(columns={"value": "demanda_mw"})

    return df



In [ ]:
#Aqui ya uso la función
prueba = descargar_semana("2026-10-08T00:00", "2026-10-14T23:59")
print(prueba.head())
print(prueba.shape)


In [ ]:
#   Por ejemplo en otra semana
enero = descargar_semana("2025-01-01T00:00", "2025-01-07T23:59")
print(enero.head())
print(enero.shape)

In [ ]:
# Ahora a los 2 años
# Creamos una lista con el lunes de cada semana, desde enero 2024 hasta hoy
# freq="7D" significa "cada 7 días"
semanas = pd.date_range(start="2024-01-01", end="2026-10-28", freq="7D")

print(len(semanas))
print(semanas[:3])   # las 3 primeras


In [ ]:
# Ahora bucle
import time

trozos = []   # lista vacía donde iremos guardando cada semana

for inicio in semanas:   # ← primero probamos con 3 semanas
    # a) Calculamos el final de la semana
    fin = inicio + pd.Timedelta(days=6, hours=23, minutes=59)

    # b) Convertimos las fechas al formato de texto que pide la API
    inicio_txt = inicio.strftime("%Y-%m-%dT%H:%M")
    fin_txt = fin.strftime("%Y-%m-%dT%H:%M")

    # c) Descargamos esa semana con nuestra función
    semana = descargar_semana(inicio_txt, fin_txt)

    # d) La añadimos a la lista
    trozos.append(semana)

    # e) Esperamos 1 segundo para no saturar la API
    time.sleep(1)

    # f) Mostramos el progreso
    print(inicio_txt, "→", fin_txt, "|", len(semana), "filas")

# Unimos todas las semanas en una sola tabla (fuera del bucle: sin sangría)
df_total = pd.concat(trozos, ignore_index=True)
print(df_total.shape)


Podmeos observar como la mayoria tiene 2016 filas pero algunas no.


* ¿Por qué las semanas de finales de marzo tienen exactamente 12 filas menos y las de finales de octubre, 12 filas más?  

Último domingo de marzo a las 02:00 los relojes saltan directamente a las 03:00.Una hora menos son 12 datos de 5 minutos menos = 2.004 filas.
Último domingo de octubre: a las 03:00 los relojes vuelven a las 02:00, así que la hora de 02:00 a 03:00 ocurre dos veces. Una hora más son 12 datos más = 2.028 filas.

* Las semanas con números "raros" (1.801, 1.884, 1.991…) no tienen nada que ver con el cambio de hora, ¿entonces que ha pasado?

Son datos que faltan en el origen





In [ ]:
# Guardamos los datos descargados en drive para no tener que volver a descargarlos
df_total.to_csv(f"{RUTA}/data/raw/demanda_ree.csv", index=False)
print("Guardado ")
